In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "b0dc7bc1-8d6c-4fd0-a0e8-5db1df8fca43",
   "metadata": {},
   "source": [
    "# Compilation and Speeding Up\n",
    "\n",
    "Python is an interpreted language.  This means that the code you write is basically being interpreted line by line (this is an oversimplification, but not far from the truth).  Each time a line of code is read, it has to be converted into equivalent machine language instructions.  For example, a `for` loop will need a register to be initialized, an instruction for incrementing the counter, an instruction to check the limits, and suitable branching statements.\n",
    "\n",
    "When a program is *compiled*, it is converted into machine language once and for all, and only that code is then run.  This also means that any change in the code requires a complete recompilation.  Compared to Python, this is less interactive and takes a longer time to do.\n",
    "\n",
    "So compiled languages pay a cost at compile time, and reap the benefits at run time.  If you expect that your program is going to run multiple times, then it is usually worth checking if this cost is worth it."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "37e6f8b6-32bc-4625-9455-c3c4d5512b3e",
   "metadata": {},
   "source": [
    "## Speed of Python\n",
    "\n",
    "Python code is typically slow for a number of reasons:\n",
    "\n",
    "- Data types are not known ahead of time, and the type of a variable can be dynamically changed.  You can store a string in a variable that previously had an `int` for example, and there will be no conflict.  This makes it hard to optimize variables as you do not know how they will change in future.\n",
    "- Semantics of certain operations are different in Python than they are in other languages or machine code.  For example, *Divide by Zero* will cause an exception to be raised in Python code.  On the other hand, in C code it will result in the program crashing.  It may be possible to catch this exception in languages like C++, but it is optional and not mandatory, so it is possible to crash as well.  Such checks add extra code and slow the program down.\n",
    "- Accessing an index that is beyond the bounds of a list will cause an Error to be raised.  In C it will not be an error, but may cause the program to crash with a Segmentation Fault.\n",
    "\n",
    "Similarly, there are other situations where the semantics of the Python code differ from a similar C or machine language representation.  Whenever this happens, there is a chance that the Python will be slower than the raw code.\n",
    "\n",
    "## Improving Speed\n",
    "\n",
    "The simplest approach for speeding things up is to try and convert the Python code to a lower level language like C, compile it, and then run the compiled code.  However, due to the above restrictions, this has to be done with care, to avoid changing the meaning of the program.\n",
    "\n",
    "## Cython\n",
    "\n",
    "*Cython* is a particular variant of the Python language: it introduces several new syntactic elements into the language to address the issues with types and compilation.  The usual way of running it is to compile the code into a dynamic library, and then import this into Python.  However, in Jupyter notebooks, there is an easier approach that can be used, which makes use of the Cython extensions and *magic annotations*."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "32863242-1fbd-4877-a00e-1581be96f196",
   "metadata": {},
   "source": [
    "# Timing and Optimization\n",
    "\n",
    "We first measure the time taken for a simple function.  Then we can look at optimizing this using Cython."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "157ec55c-0a02-4ded-a199-af6e708e00e3",
   "metadata": {},
   "outputs": [],
   "source": [
    "def isPrime(n):\n",
    "    for i in range(2,int(n**0.5)+1):\n",
    "        if n%i==0:\n",
    "            return False\n",
    "        \n",
    "    return True"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e2939122-e8da-4f90-b198-1be2c38b2b0c",
   "metadata": {},
   "outputs": [],
   "source": [
    "%timeit isPrime(999999937)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2cdff68a-607f-4032-9e53-ce6812f01230",
   "metadata": {},
   "source": [
    "## Cython\n",
    "\n",
    "First we just apply cython without any optimizations.  Later we will see the effect of adding the optimizations to it."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "199d5040-7961-46e7-a380-837b7a022133",
   "metadata": {},
   "outputs": [],
   "source": [
    "%load_ext Cython"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ee4f0740-e914-4443-b6c0-e5b1ad261bf8",
   "metadata": {},
   "outputs": [],
   "source": [
    "%%cython --annotate\n",
    "\n",
    "def cbasic_isPrime(n):\n",
    "    for i in range(2,int(n**0.5)+1):\n",
    "        if n%i==0:\n",
    "            return False\n",
    "        \n",
    "    return True"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e86d216a-7f4f-4ea0-ba27-3918836118e8",
   "metadata": {},
   "outputs": [],
   "source": [
    "%timeit cbasic_isPrime(999999937)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a7d9f22b-3952-4883-831f-cbeb1c6e3045",
   "metadata": {},
   "source": [
    "### Optimized\n",
    "\n",
    "Now apply several optimizations.  In the code below, the actual optimizations are commented out.  Try uncommenting them one by one to try and see which has the biggest impact on the result.\n",
    "\n",
    "In general, you need to look for the following:\n",
    "\n",
    "- where is the bulk of the time being spent - most likely it is inside loops.  Here it is the `for` loop\n",
    "    - To handle this, we explicitly declare `i` as an integer: in fact, as a `cdef int`: this means a C type integer.  Try commenting that line out to see how it changes the result.\n",
    "- what kind of data types are being used?  C prefers to use data types close to what the computer has: for example 32-bit int, 32-bit single precision float etc.  These are highly optimized.  Python on the other hand naturally tries to accommodate larger integers if needed, but the cost of that is additional checks for overflow.  If you force it to use C data types, it will remove some of the checks.\n",
    "    - Which are all the lines here \n",
    "- what kind of operations are being used?  For example, C type division will work only within the precision of the numbers you are using, and will round off or give wrong answers if the numbers are out of range, rather than trying to catch the errors and report them.  This makes it possible to have severe errors, but if you know your values cannot fall in the error regions, you can use this."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2487e566-d72f-4066-90ce-4532a4e9738e",
   "metadata": {},
   "outputs": [],
   "source": [
    "%%cython --annotate\n",
    "\n",
    "import cython\n",
    "\n",
    "# @cython.cdivision(True)\n",
    "def c_isPrime(int n):\n",
    "    # cdef int i\n",
    "    # cdef float sqrtn = (n**0.5)\n",
    "    # cdef int lim = int(sqrtn)+1\n",
    "    # Note: if you uncomment the above two lines, then comment out the one below\n",
    "    lim = int(n**0.5) + 1\n",
    "    for i in range(2,lim):\n",
    "        if n%i==0:\n",
    "            return False\n",
    "        \n",
    "    return True"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "35034636-6d73-4640-b463-b9520381faf6",
   "metadata": {},
   "outputs": [],
   "source": [
    "# %timeit c_isPrime(999999999)\n",
    "%timeit c_isPrime(999999937)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "27ba4c15",
   "metadata": {},
   "source": [
    "# Matrix multiplication\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "55baf924",
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "def matrix_multiply(u, v):\n",
    "    m, n = u.shape\n",
    "    n, p = v.shape\n",
    "    res = np.zeros((m, p))\n",
    "    for i in range(m):\n",
    "        for j in range(p):\n",
    "            res[i,j] = 0\n",
    "            for k in range(n):\n",
    "                res[i,j] += u[i,k] * v[k,j]\n",
    "    return res\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "8e820ced",
   "metadata": {},
   "outputs": [],
   "source": [
    "u = np.random.random((100,100))\n",
    "v = np.random.random((100,100))\n",
    "# %timeit -n 100 -r 3 matrix_multiply(u,v)\n",
    "# %timeit matrix_multiply(u, v)\n",
    "# %timeit np.matmul(u, v)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d84b4375-e253-44e8-ae16-67e90183ec11",
   "metadata": {},
   "source": [
    "## Optimized Matrix Multiply\n",
    "\n",
    "Can we apply the same techniques to speed up matrix multiply in Python?  Consider all the places where changes would make sense.  In addition, there are a couple of decorators that also help to speed things up by avoiding extra checks on the array sizes."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "32f6832e-82a0-49d2-a654-4d49c2a9c90e",
   "metadata": {},
   "outputs": [],
   "source": [
    "%load_ext Cython"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "80a7120b",
   "metadata": {},
   "outputs": [],
   "source": [
    "%%cython -a\n",
    "\n",
    "import numpy as np\n",
    "import cython\n",
    "\n",
    "@cython.boundscheck(False)\n",
    "@cython.wraparound(False)\n",
    "def cy_matmul(float[:,:] u, float[:,:] v, float[:,:] res):\n",
    "# def cy_matmul(u, v, res):\n",
    "    cdef int m, n, p\n",
    "    cdef int i, j, k\n",
    "    m = u.shape[0]\n",
    "    n = u.shape[1]\n",
    "    p = v.shape[1]\n",
    "    res = np.zeros((m, p), dtype=np.float32)\n",
    "    for i in range(m):\n",
    "        for j in range(p):\n",
    "            res[i,j] = 0\n",
    "            for k in range(n):\n",
    "                res[i,j] += u[i,k] * v[k,j]\n",
    "    return res\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f132b3b7-5962-4a5f-9d99-968d9f83ef39",
   "metadata": {},
   "outputs": [],
   "source": [
    "# u = np.float32(np.random.random((100,1000)))\n",
    "# v = np.float32(np.random.random((1000,100)))\n",
    "# res = np.zeros((100, 100), dtype=np.float32)\n",
    "# %timeit cy_matmul(u, v, res)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "42cfb37c-946e-43a8-bd23-80f2528e8cfd",
   "metadata": {},
   "source": [
    "## Performance testing\n",
    "\n",
    "Try iterating this across different combinations of matrix sizes to see how the time varies."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "343f0170-8c5c-447c-a895-565105f61cfd",
   "metadata": {},
   "outputs": [],
   "source": [
    "M, N, P = 100, 100, 100\n",
    "u = np.float32(np.random.random((M, N)))\n",
    "v = np.float32(np.random.random((N, P)))\n",
    "res = np.zeros((M, P), dtype=np.float32)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "85abd61a-c22d-443d-a771-8c312b72cf78",
   "metadata": {},
   "outputs": [],
   "source": [
    "%timeit matrix_multiply(u, v)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "06fc9cf6-a41b-4f67-930d-554e254d179b",
   "metadata": {},
   "outputs": [],
   "source": [
    "%timeit cy_matmul(u, v, res)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7292bdd7-b329-4ac8-8324-13d4499be792",
   "metadata": {},
   "outputs": [],
   "source": [
    "%timeit np.matmul(u, v)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "57102969-007b-4545-88fe-0a9b088bfecd",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.2"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}